<a href="https://colab.research.google.com/github/0MSA0/supreme-robot/blob/main/notebooks/Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Login**

In [ ]:
import wandb

wandb.login()

# Project that the run is recorded to
project = "Supreme Training"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Init**

In [ ]:
import tiktoken
import torch

# Get device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print(f"Using {device} device.")

# Create tokenize
tokenizer = tiktoken.get_encoding("gpt2")

# Training settings
config = {
    "epochs" : 1,
    "lr" : 0.0005,
    "batch_size": 20,
    "weight_decay": 0.1,
    "data_loader_num_workers": 1,
}

# Model settings
gpt_config_124_m = {
  "vocab_size": tokenizer.n_vocab,  # Vocabulary size
  "context_length": 512,  # Context length
  "emb_dim": 768,  # Embedding dimension
  "n_heads": 12,  # Number of attention heads
  "n_layers": 12,  # Number of layers
  "temperature": 1.0,  # Temperature
  "drop_rate": 0.1,  # Dropout rate
  "qkv_bias": False,  # Query-Key-Value bias
}

save_path = "/content/drive/MyDrive/SOTA2/model_and_optimizer.pth"

Using cpu device.


**Getting the data**

In [ ]:
from datasets import Dataset, load_dataset
from torch.utils.data import DataLoader
from supreme_robot import StreamingSupremeDataset

data_stream = load_dataset(
    "Salesforce/wikitext",
    name="wikitext-103-raw-v1",
    split="train",
    streaming=True
    ).shuffle(seed=69420, buffer_size=10000)

streaming_dataset = StreamingSupremeDataset(
    hf_stream=data_stream,
    tokenizer=tokenizer,
    max_length=gpt_config_124_m["context_length"],
    stride=gpt_config_124_m["context_length"]
)

dataloader = DataLoader(
    streaming_dataset,
    batch_size=config["batch_size"],
    drop_last=True,
    num_workers=0
)



**Train**

In [ ]:
from supreme_robot import SupremeGPTModel

model = SupremeGPTModel(gpt_config_124_m)
model.to(device)

loss_fn = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["lr"],
    weight_decay=config["weight_decay"]
)

with wandb.init(project=project,config=config) as run:
  global_step = 0
  for epoch in range(config["epochs"]):
      model.train()
      for step, (in_batch, out_batch) in enumerate(dataloader):
        # Send data to device
        in_batch = in_batch.to(device)
        out_batch = out_batch.to(device)

        optimizer.zero_grad()

        # Forward pass
        out_predicted = model(in_batch)

        logits = out_predicted.view(-1, out_predicted.size(-1))

        # Targets: [batch_size * seq_length]
        targets = out_batch.view(-1)

        # Compute loss
        loss = loss_fn(logits, targets)

        # Back propagation
        loss.backward()

        optimizer.step()

        # Don't use grad for accuracy
        with torch.no_grad():
                # Compute accuracy
                predictions = torch.argmax(logits, dim=-1)
                accuracy = (predictions == targets).float().mean().item()

        # Get VRAM
        vram_gb = torch.cuda.memory_allocated(device) / (1024 ** 3)

        # LOG
        run.log({
          "train/loss": loss.item(),
          "train/accuracy": accuracy,
          "system/vram_allocated_gb": vram_gb,
          "epoch": epoch,
          "batch_step": step
        }, step=global_step)

        global_step += 1

run.finish()

# Save weights and optimizer
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    },
    save_path
)



**Testing**

In [ ]:
from supreme_robot import SupremeGPTModel
import textwrap

load_path = "/content/drive/MyDrive/SOTA2/first_model_and_optimizer_20261112-0227.pth"

start_context = "Germany has"

def test_generate_text(model, idx, max_new_tokens, context_size, top_k=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k, dim=-1)
            min_val = top_logits[:, [-1]]
            logits = torch.where(
                logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits
            )

        probas = torch.softmax(logits, dim=-1)

        if top_k is not None:
            idx_next = torch.multinomial(probas, num_samples=1)
        else:
            idx_next = torch.argmax(probas, dim=-1, keepdim=True)

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

# Load saved weights from gdrive
weights = torch.load(load_path, weights_only=True, map_location="cpu")

trained_model = SupremeGPTModel(gpt_config_124_m)
trained_model.load_state_dict(weights["model_state_dict"])
trained_model.eval()

# Generate text
out_trained = test_generate_text(
        model=trained_model,
        idx=encoded_tensor,
        max_new_tokens=50,
        context_size=gpt_config_124_m["context_length"],
        top_k=50,
    )

decoded_text_trained = tokenizer.decode(out_trained.squeeze(0).tolist())
print("Input: ", start_context, "\n")
print("Output:\n" + "\n".join(textwrap.wrap(decoded_text_trained, width=80)))


